In [27]:
import pandas as pd
import numpy as np
import openpyxl as xl

In [28]:
from pathlib import Path
import gc

import pandas as pd

leads_dir = Path(r"D:\dataropes\b2b_ecommerce_project\data\output\leads")

# Skip Excel temp/lock files (~$...) that are created while a file is open
excel_files = sorted(
    p for p in leads_dir.glob("*.xlsx")
    if not p.name.startswith("~$")
)

if not excel_files:
    raise FileNotFoundError(f"No .xlsx files found in {leads_dir}")

# Read one file at a time and keep only the resulting frame (no extra copies)
frames = []
failed = []
for excel_file in excel_files:
    try:
        frame = pd.read_excel(excel_file, engine="openpyxl")
    except Exception as e:
        failed.append((excel_file.name, repr(e)))
        continue
    if not frame.empty:
        frames.append(frame)

# Single concat at the end (repeated concat inside the loop would copy the data every time)
df = pd.concat(frames, ignore_index=True)

del frames
gc.collect()

print(f"Files found: {len(excel_files)} | merged: {len(excel_files) - len(failed)} | failed: {len(failed)}")
for name, err in failed:
    print(f"  FAILED {name}: {err}")
print(f"Merged DataFrame shape: {df.shape}")

df.head()

Files found: 560 | merged: 560 | failed: 0
Merged DataFrame shape: (1161, 44)


,lead_id,created_date,source,campaign_name,utm_source,utm_medium,utm_campaign,country,city,company_name,...,Unnamed: 12,Unnamed: 13,Unnamed: 14,Unnamed: 15,Unnamed: 16,Unnamed: 17,Unnamed: 18,Unnamed: 19,Unnamed: 20,Unnamed: 21
0,LD-20241031-001,2024-10-31 00:00:00,Email,Newsletter Promo,email,email,newsletter_promo,Argentina,Tucumán,Condor Metalurgica S.R.L.,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,LD-20241101-001,01/11/2024,Google Ads,Retargeting Display,google_ads,display,retargeting_display,Brazil,São Paulo,Sol Transportes S.A.,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,lead_score,estimated_order_volume,status,utm_content,contact_name,contact_email,phone,job_title,sales_rep,last_contacted_date
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,43,5702,Contacted,banner_b,Juan Sanchez,juan.sanchez@riometalurgica.com,+54 77 7996-4634,Procurement Analyst,Paula Ibarra,2024-11-04 00:00:00


In [24]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1161 entries, 0 to 1160
Data columns (total 44 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   lead_id                 1098 non-null   str    
 1   created_date            1098 non-null   object 
 2   source                  1098 non-null   str    
 3   campaign_name           1098 non-null   str    
 4   utm_source              1098 non-null   str    
 5   utm_medium              1098 non-null   str    
 6   utm_campaign            1098 non-null   str    
 7   country                 1098 non-null   str    
 8   city                    1046 non-null   object 
 9   company_name            1098 non-null   str    
 10  company_size            1098 non-null   str    
 11  industry                1098 non-null   str    
 12  lead_score              1043 non-null   float64
 13  estimated_order_volume  1073 non-null   object 
 14  status                  1098 non-null   str    
 15

In [29]:
from collections import Counter
from pathlib import Path

import pandas as pd
from openpyxl import load_workbook

leads_dir = Path(r"D:\dataropes\b2b_ecommerce_project\data\output\leads")
excel_files = sorted(
    p for p in leads_dir.glob("*.xlsx")
    if not p.name.startswith("~$")
)

SCAN_ROWS = 10  # only the top rows are read, so this stays light on memory


def top_rows(path):
    """First SCAN_ROWS rows of the first sheet, exactly as stored (no header applied)."""
    return pd.read_excel(path, engine="openpyxl", header=None, nrows=SCAN_ROWS)


def row_values(row):
    """Non-empty cell values of a row, as a tuple of strings."""
    return tuple(str(v).strip() for v in row if pd.notna(v))


def total_rows(path):
    """Total sheet rows without loading the data (read-only mode)."""
    wb = load_workbook(path, read_only=True)
    try:
        return wb.worksheets[0].max_row
    finally:
        wb.close()


# 1) Read the top rows of every file once
tops = {}
unreadable = []
for f in excel_files:
    try:
        tops[f] = top_rows(f)
    except Exception as e:
        unreadable.append((f.name, repr(e)))

# 2) Reference header = the most common first row across all files
reference_header = Counter(
    row_values(t.iloc[0]) for t in tops.values() if len(t)
).most_common(1)[0][0]

# 3) Flag every file whose first row is not that header
records = []
for f, t in tops.items():
    if row_values(t.iloc[0]) == reference_header:
        continue

    header_row = next(
        (i for i in range(len(t)) if row_values(t.iloc[i]) == reference_header),
        None,
    )
    rows_above = [] if header_row is None else [
        row_values(t.iloc[i]) for i in range(header_row)
    ]
    records.append({
        "file": f.name,
        "header_found_on_excel_row": None if header_row is None else header_row + 1,
        "rows_above_header": None if header_row is None else header_row,
        "text_above_header": None if header_row is None else [r for r in rows_above if r],
        "sheet_rows_total": total_rows(f),
        "data_rows": None if header_row is None else total_rows(f) - header_row - 1,
        "first_row_found": row_values(t.iloc[0]),
    })

misaligned = pd.DataFrame(records)

print(f"Reference header ({len(reference_header)} cols): {list(reference_header)}")
print(f"Files scanned: {len(tops)} | first row is NOT the header: {len(misaligned)} | unreadable: {len(unreadable)}")
for name, err in unreadable:
    print(f"  UNREADABLE {name}: {err}")

pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 250)
misaligned

Reference header (22 cols): ['lead_id', 'created_date', 'source', 'campaign_name', 'utm_source', 'utm_medium', 'utm_campaign', 'country', 'city', 'company_name', 'company_size', 'industry', 'lead_score', 'estimated_order_volume', 'status', 'utm_content', 'contact_name', 'contact_email', 'phone', 'job_title', 'sales_rep', 'last_contacted_date']
Files scanned: 560 | first row is NOT the header: 15 | unreadable: 0


,file,header_found_on_excel_row,rows_above_header,text_above_header,sheet_rows_total,data_rows,first_row_found
0,leads_2024-11-02.xlsx,3,2,"[(Marketing leads export,)]",5,2,"(Marketing leads export,)"
1,leads_2025-02-11.xlsx,3,2,"[(Marketing leads export,)]",4,1,"(Marketing leads export,)"
2,leads_2025-03-08.xlsx,3,2,"[(Marketing leads export,)]",4,1,"(Marketing leads export,)"
3,leads_2025-08-06.xlsx,3,2,"[(Marketing leads export,)]",5,2,"(Marketing leads export,)"
4,leads_2025-08-22.xlsx,3,2,"[(Marketing leads export,)]",7,4,"(Marketing leads export,)"
5,leads_2026-01-06.xlsx,3,2,"[(Marketing leads export,)]",5,2,"(Marketing leads export,)"
6,leads_2026-03-23.xlsx,3,2,"[(Marketing leads export,)]",5,2,"(Marketing leads export,)"
7,leads_2026-03-25.xlsx,3,2,"[(Marketing leads export,)]",5,2,"(Marketing leads export,)"
8,leads_2026-04-14.xlsx,3,2,"[(Marketing leads export,)]",7,4,"(Marketing leads export,)"
9,leads_2026-07-19.xlsx,3,2,"[(Marketing leads export,)]",4,1,"(Marketing leads export,)"


## 0. Load the complete leads dataset
Read-only. Builds `leads_raw` (all files, exactly as stored, header row detected per file). Every section below uses it; the existing `df` is not touched.

In [30]:
# =====================================================================
# SECTION 0 - LOAD THE COMPLETE LEADS DATASET (read-only)
# Every cell below only READS this data. Nothing on disk is modified, and the
# existing `df` of the merge cell is left untouched: this cell builds its own
# frame called `leads_raw`.
#   * every sheet is read exactly as stored (header=None, dtype=object), so no value is
#     converted, cleaned or dropped -> the profiling below sees the true raw values
#   * the real header row is detected per file (15 files have a title row above it)
#   * two lineage columns are added: _source_file and _source_row (Excel row number)
# =====================================================================
import re
from collections import Counter
from datetime import datetime
from pathlib import Path

import pandas as pd

LEADS_DIR = Path(r"D:\dataropes\b2b_ecommerce_project\data\output\leads")

# Excel lock files (~$...) are not real workbooks
lead_files = sorted(p for p in LEADS_DIR.glob("*.xlsx") if not p.name.startswith("~$"))


def _row_vals(row):
    """Non-empty cell values of one row as a tuple of stripped strings."""
    return tuple(str(v).strip() for v in row if pd.notna(v))


# 1) Read every sheet untouched
raw_sheets = {f.name: pd.read_excel(f, engine="openpyxl", header=None, dtype=object) for f in lead_files}

# 2) The correct header = the most common first row across all files (data-driven, nothing assumed)
REFERENCE_HEADER = Counter(
    _row_vals(s.iloc[0]) for s in raw_sheets.values() if len(s)
).most_common(1)[0][0]

# 3) Cut every sheet at its real header row and stack them
frames, file_info = [], []
for f in lead_files:
    sheet = raw_sheets[f.name]
    header_idx = next(
        (i for i in range(min(len(sheet), 10)) if _row_vals(sheet.iloc[i]) == REFERENCE_HEADER), None
    )
    info = {
        "file": f.name,
        "file_date": pd.to_datetime(re.search(r"\d{4}-\d{2}-\d{2}", f.name).group(), errors="coerce"),
        "size_kb": round(f.stat().st_size / 1024, 1),
        "last_modified": datetime.fromtimestamp(f.stat().st_mtime),
        "header_on_excel_row": None if header_idx is None else header_idx + 1,
        "data_rows": 0,
    }
    if header_idx is not None:
        header_row = sheet.iloc[header_idx]
        keep = header_row.notna().values                       # keep only the columns that have a header name
        body = sheet.iloc[header_idx + 1:, keep].copy()
        body.columns = [str(v).strip() for v in header_row[keep]]
        body = body.dropna(how="all")                          # drop fully empty rows
        body.insert(0, "_source_file", f.name)                 # lineage: which file the row came from
        body.insert(1, "_source_row", body.index + 1)          # lineage: Excel row number inside that file
        info["data_rows"] = len(body)
        frames.append(body)
    file_info.append(info)

leads_raw = pd.concat(frames, ignore_index=True)
file_info = pd.DataFrame(file_info)

# Quick confirmation
LEAD_COLUMNS = [c for c in leads_raw.columns if not c.startswith("_")]   # the 22 business columns
print(f"Files read          : {len(lead_files)}")
print(f"Rows loaded         : {len(leads_raw):,}")
print(f"Business columns    : {len(LEAD_COLUMNS)}")
print(f"Files w/o header    : {int(file_info['header_on_excel_row'].isna().sum())}")
leads_raw.head(3)

Files read          : 560
Rows loaded         : 1,131
Business columns    : 22
Files w/o header    : 0


,_source_file,_source_row,lead_id,created_date,source,campaign_name,utm_source,utm_medium,utm_campaign,country,...,lead_score,estimated_order_volume,status,utm_content,contact_name,contact_email,phone,job_title,sales_rep,last_contacted_date
0,leads_2024-10-31.xlsx,2,LD-20241031-001,2024-10-31 00:00:00,Email,Newsletter Promo,email,email,newsletter_promo,Argentina,...,42,"$1,540",Contacted,text_ad,Carlos Silva,carlos.silva@condormetalurgica.com,+54 66 3674-9379,Purchasing Manager,Marcos Vidal,2024-11-03 00:00:00
1,leads_2024-11-01.xlsx,2,LD-20241101-001,01/11/2024,Google Ads,Retargeting Display,google_ads,display,retargeting_display,Brazil,...,35,12318,Qualified,banner_b,Facundo Gomez,facundo.gomez@soltransportes.com,NaN,CEO,Luis Benitez,2024-11-02 00:00:00
2,leads_2024-11-02.xlsx,4,LD-20241102-001,2024-11-02 00:00:00,Google Ads,Retargeting Display,google_ads,display,retargeting_display,Argentina,...,43,5702,Contacted,banner_b,Juan Sanchez,juan.sanchez@riometalurgica.com,+54 77 7996-4634,Procurement Analyst,Paula Ibarra,2024-11-04 00:00:00


## 1. Metadata
*Data about the data*: what a row is, where it comes from, size, time coverage, structure and basic health.

In [31]:
# =====================================================================
# SECTION 1 - METADATA  ("data about the data")
# Structure, size, source, time coverage, grain and overall health of the leads dataset.
# =====================================================================
from openpyxl import load_workbook

# ---- 1.1 What one row represents and where the data comes from -------------------------------
# (documented facts: gen_leads.py + docs/data_dictionary_and_kpi_mapping.md)
dataset_facts = {
    "Dataset name": "Marketing leads",
    "What one row represents": "ONE marketing lead (a prospect company + contact) at the moment it was created",
    "Grain": "1 row = 1 lead. A lead should appear once, in the file of its creation day",
    "Primary key": "lead_id  (format LD-YYYYMMDD-NNN)",
    "File format": "Excel workbook (.xlsx), one file per day that has leads",
    "File naming": "leads_YYYY-MM-DD.xlsx   (the date = lead creation day)",
    "Location": str(LEADS_DIR),
    "Produced by": "data_generation/gen_leads.py (synthetic marketing export), run by Airflow DAG b2b_daily_pipeline",
    "Refresh": "Daily (new file per day, none on days without leads); full 2-year backfill on first run",
}
print("1.1 DATASET FACTS")
print(pd.Series(dataset_facts).to_string(), "\n")

# ---- 1.2 Physical metadata (measured from the files) ------------------------------------------
sheet_names = Counter()
for f in lead_files:                                   # read-only mode: sheet names without loading the data
    wb = load_workbook(f, read_only=True)
    sheet_names.update(wb.sheetnames)
    wb.close()

file_dates = pd.to_datetime(file_info["file_date"])
all_days = pd.date_range(file_dates.min(), file_dates.max())
missing_days = all_days.difference(file_dates)

physical = {
    "Number of files": len(lead_files),
    "Sheet name(s) (files per sheet)": dict(sheet_names),
    "Total size on disk (MB)": round(file_info["size_kb"].sum() / 1024, 2),
    "Rows per file (min / median / max)": f"{file_info['data_rows'].min()} / {file_info['data_rows'].median():.0f} / {file_info['data_rows'].max()}",
    "Total rows": len(leads_raw),
    "Business columns": len(LEAD_COLUMNS),
    "Rows x columns in memory": leads_raw.shape,
    "Memory used by leads_raw (MB)": round(leads_raw.memory_usage(deep=True).sum() / 1024**2, 2),
    "First file date": file_dates.min().date(),
    "Last file date": file_dates.max().date(),
    "Calendar days in that span": len(all_days),
    "Days WITH a file": len(file_dates),
    "Days WITHOUT a file": len(missing_days),
    "Files with a title row above the header": int((file_info["header_on_excel_row"] > 1).sum()),
}
print("1.2 PHYSICAL METADATA")
print(pd.Series(physical).to_string(), "\n")

# ---- 1.3 Structural metadata (consistency checks) ---------------------------------------------
n_distinct_ids = leads_raw["lead_id"].nunique()
exact_dups = leads_raw.duplicated(subset=LEAD_COLUMNS, keep="first").sum()
id_in_many_files = (leads_raw.groupby("lead_id")["_source_file"].nunique() > 1).sum()
# the date embedded in lead_id (LD-YYYYMMDD-NNN) should equal the file date
id_date = pd.to_datetime(leads_raw["lead_id"].astype(str).str.extract(r"LD-(\d{8})-")[0], format="%Y%m%d", errors="coerce")
file_date_per_row = pd.to_datetime(leads_raw["_source_file"].str.extract(r"(\d{4}-\d{2}-\d{2})")[0], errors="coerce")

empty_cells = int(leads_raw[LEAD_COLUMNS].isna().sum().sum())
structural = {
    "Distinct lead_id": n_distinct_ids,
    "Rows whose lead_id repeats an earlier row": len(leads_raw) - n_distinct_ids,
    "Fully identical duplicate rows (all 22 columns)": int(exact_dups),
    "lead_id found in more than one file": int(id_in_many_files),
    "Rows where date inside lead_id != file date": int((id_date != file_date_per_row).sum()),
    "Rows with a missing lead_id": int(leads_raw["lead_id"].isna().sum()),
    "Total empty cells (business columns)": empty_cells,
    "Share of cells empty (%)": round(empty_cells / leads_raw[LEAD_COLUMNS].size * 100, 2),
}
print("1.3 STRUCTURAL METADATA")
print(pd.Series(structural, dtype=object).to_string(), "\n")   # dtype=object keeps whole numbers as integers

# ---- 1.4 Volume over time and the days that have no file ---------------------------------------
print("1.4 ROWS PER MONTH")
print(leads_raw.groupby(file_date_per_row.dt.to_period("M")).size().rename("rows").to_string(), "\n")
print("1.4b DAYS WITH NO FILE (first 40):", [d.date().isoformat() for d in missing_days[:40]])

1.1 DATASET FACTS
Dataset name                                                                               Marketing leads
What one row represents     ONE marketing lead (a prospect company + contact) at the moment it was created
Grain                           1 row = 1 lead. A lead should appear once, in the file of its creation day
Primary key                                                              lead_id  (format LD-YYYYMMDD-NNN)
File format                                        Excel workbook (.xlsx), one file per day that has leads
File naming                                         leads_YYYY-MM-DD.xlsx   (the date = lead creation day)
Location                                              D:\dataropes\b2b_ecommerce_project\data\output\leads
Produced by                data_generation/gen_leads.py (synthetic marketing export), run by Airflow DA...
Refresh                    Daily (new file per day, none on days without leads); full 2-year backfill o... 

1.2 PHYSICAL META

## 2. Data dictionary
What every column means, its expected format, and what the data really contains.

In [32]:
# =====================================================================
# SECTION 2 - DATA DICTIONARY
# For every column: meaning, expected format / allowed values, and what the data REALLY contains
# (types found, empties, distinct values, ranges, samples and how many values break the expected format).
# =====================================================================

# ---- 2.1 Documented meaning + expected format (from gen_leads.py / common.py / project docs) ----
# column -> (business meaning, expected type, expected format / allowed values)
EXPECTED = {
    "lead_id":                ("Unique identifier of the lead (primary key)", "text", "LD-YYYYMMDD-NNN"),
    "created_date":           ("Day the lead was created (= the file date)", "date", "date; should not be after the file date"),
    "source":                 ("Marketing channel the lead came from", "text", "Google Ads, LinkedIn, Trade Show, Referral, Facebook, Email, Organic"),
    "campaign_name":          ("Marketing campaign that generated the lead", "text", "8 campaigns (most reliable campaign key)"),
    "utm_source":             ("UTM source tag = slug of the source", "text", "lower_snake_case"),
    "utm_medium":             ("UTM medium tag (type of traffic)", "text", "cpc, paid_social, event, referral, display, email, organic"),
    "utm_campaign":           ("UTM campaign tag = slug of campaign_name", "text", "lower_snake_case"),
    "country":                ("Country of the lead company", "text", "Argentina, Chile, Uruguay, Brazil (full English name)"),
    "city":                   ("City of the lead company", "text", "free text; may be empty"),
    "company_name":           ("Name of the prospect company", "text", "free text, ends with a legal suffix (S.A., S.R.L., S.A.S.)"),
    "company_size":           ("Size band of the company (employees)", "text", "1-10, 11-50, 51-200, 201-500, 500+"),
    "industry":               ("Industry of the company", "text", "9 industries"),
    "lead_score":             ("Lead quality score (higher = better lead)", "integer", "whole number 1-100"),
    "estimated_order_volume": ("Expected order value of the lead (currency units)", "integer", "positive whole number"),
    "status":                 ("Funnel stage of the lead (snapshot at creation, never updated)", "text", "New, Contacted, Qualified, Proposal, Lost, Won"),
    "utm_content":            ("UTM content tag = which creative/ad variant", "text", "banner_a, banner_b, text_ad; may be empty"),
    "contact_name":           ("Full name of the contact person", "text", "First Last"),
    "contact_email":          ("Contact e-mail (for Won leads = the customer's login e-mail = join key)", "text", "valid e-mail, lower case"),
    "phone":                  ("Contact phone number", "text", "+CC [9] AA NNNN-NNNN"),
    "job_title":              ("Job title of the contact", "text", "6 titles (e.g. Purchasing Manager)"),
    "sales_rep":              ("Sales representative that owns the lead", "text", "4 reps"),
    "last_contacted_date":    ("Last day the sales rep contacted the lead", "date", "date; empty for status New"),
}


# ---- 2.2 Helpers that read a raw value without changing the data ----------------------------------
def is_text(v):
    return isinstance(v, str)


def to_number(v):
    """Raw value -> float (strips '$' and ',' from text like '$4,321'); NaN if not numeric."""
    if isinstance(v, (int, float)) and not isinstance(v, bool):
        return float(v)
    if isinstance(v, str):
        try:
            return float(re.sub(r"[$,\s]", "", v))
        except ValueError:
            return float("nan")
    return float("nan")


def to_date(v):
    """Raw value -> Timestamp. Handles Excel datetimes and text dates written dd/mm/yyyy."""
    if isinstance(v, (datetime, pd.Timestamp)):
        return pd.Timestamp(v)
    if isinstance(v, str):
        for fmt in ("%d/%m/%Y", "%Y-%m-%d", "%Y-%m-%d %H:%M:%S"):
            try:
                return pd.Timestamp(datetime.strptime(v.strip(), fmt))
            except ValueError:
                pass
    return pd.NaT


def canonical_spelling(series):
    """Most frequent spelling per case/space-insensitive value, used to spot inconsistent casing."""
    s = series.dropna().astype(str)
    key = s.str.strip().str.lower()
    return s.groupby(key).agg(lambda x: x.value_counts().index[0])


file_date_per_row = pd.to_datetime(leads_raw["_source_file"].str.extract(r"(\d{4}-\d{2}-\d{2})")[0], errors="coerce")
PHONE_RE = re.compile(r"^\+\d{2,3} (9 )?\d{2} \d{4}-\d{4}$")
EMAIL_RE = re.compile(r"^[^@\s]+@[^@\s]+\.[^@\s]+$")


# ---- 2.3 "How many values break the expected format?" rule per column --------------------------
def format_violations(col, s):
    """Return (count, short description) of values that do not follow the expected format."""
    nn = s.dropna()
    if col == "lead_id":
        return int((~nn.astype(str).str.match(r"^LD-\d{8}-\d{3}$")).sum()), "does not match LD-YYYYMMDD-NNN"
    if col == "created_date":
        as_text = int(nn.map(is_text).sum())
        future = int((nn.map(to_date) > file_date_per_row[nn.index]).sum())
        return as_text + future, f"{as_text} stored as text (dd/mm/yyyy) + {future} dated after the file date"
    if col in ("source", "country"):
        canon = canonical_spelling(nn)
        bad = nn.astype(str) != nn.astype(str).str.strip().str.lower().map(canon)
        short = int((nn.astype(str).str.strip().str.len() <= 2).sum()) if col == "country" else 0
        return int(bad.sum()) + (short if col == "country" else 0), "inconsistent casing / extra spaces" + (" / ISO code instead of name" if col == "country" else "")
    if col == "lead_score":
        num = nn.map(to_number)
        return int(num.isna().sum() + ((num < 1) | (num > 100)).sum()), "not numeric, or outside 1-100"
    if col == "estimated_order_volume":
        return int(nn.map(is_text).sum()), "stored as text such as '$4,321'"
    if col == "contact_email":
        t = nn.astype(str)
        return int((~t.str.match(EMAIL_RE) | (t != t.str.lower().str.strip())).sum()), "invalid e-mail, upper case or padded"
    if col == "phone":
        return int((~nn.astype(str).str.match(PHONE_RE)).sum()), "not in '+CC AA NNNN-NNNN' layout"
    if col == "company_name":
        t = nn.astype(str)
        return int(((t != t.str.strip()) | t.str.isupper()).sum()), "trailing spaces or ALL UPPER CASE"
    if col == "last_contacted_date":
        return int(nn.map(is_text).sum()), "stored as text"
    if EXPECTED[col][1] == "text":
        t = nn.astype(str)
        return int((t != t.str.strip()).sum()), "leading / trailing spaces"
    return 0, "-"


# ---- 2.4 Build the dictionary -------------------------------------------------------------------
rows = []
for pos, col in enumerate(LEAD_COLUMNS, start=1):
    s = leads_raw[col]
    nn = s.dropna()
    meaning, exp_type, exp_format = EXPECTED.get(col, ("(not documented)", "?", "?"))
    n_bad, bad_desc = format_violations(col, s) if col in EXPECTED else (0, "-")

    num = nn.map(to_number)
    dates = nn.map(to_date)
    value_range = "-"
    if len(nn) and num.notna().mean() > 0.5:
        value_range = f"{num.min():,.0f} .. {num.max():,.0f}  (mean {num.mean():,.1f})"
    elif len(nn) and dates.notna().mean() > 0.5:
        value_range = f"{dates.min().date()} .. {dates.max().date()}"

    distinct = nn.astype(str).str.strip().nunique()
    rows.append({
        "#": pos,
        "column": col,
        "meaning": meaning,
        "expected_type": exp_type,
        "expected_format": exp_format,
        "types_found": ", ".join(f"{k}:{v}" for k, v in Counter(type(v).__name__ for v in nn).most_common()),
        "non_null": len(nn),
        "null": int(s.isna().sum()),
        "null_%": round(s.isna().mean() * 100, 2),
        "distinct": distinct,
        "value_range": value_range,
        "allowed_values_found": dict(nn.astype(str).str.strip().value_counts().head(12)) if distinct <= 12 else "(too many to list)",
        "examples": list(nn.astype(str).str.strip().drop_duplicates().head(3)),
        "format_violations": n_bad,
        "violation_rule": bad_desc if n_bad else "-",
    })

data_dictionary = pd.DataFrame(rows).set_index("#")

pd.set_option("display.max_colwidth", 70)
pd.set_option("display.width", 250)
pd.set_option("display.max_columns", None)
data_dictionary[["column", "meaning", "expected_type", "expected_format", "types_found", "null", "null_%", "distinct", "format_violations"]]

,column,meaning,expected_type,expected_format,types_found,null,null_%,distinct,format_violations
#,,,,,,,,,
1,lead_id,Unique identifier of the lead (primary key),text,LD-YYYYMMDD-NNN,str:1131,0,0.00,1104,0
2,created_date,Day the lead was created (= the file date),date,date; should not be after the file date,"datetime:1071, str:60",0,0.00,604,81
3,source,Marketing channel the lead came from,text,"Google Ads, LinkedIn, Trade Show, Referral, Facebook, Email, Organic",str:1131,0,0.00,21,93
4,campaign_name,Marketing campaign that generated the lead,text,8 campaigns (most reliable campaign key),str:1131,0,0.00,8,0
5,utm_source,UTM source tag = slug of the source,text,lower_snake_case,str:1131,0,0.00,7,0
6,utm_medium,UTM medium tag (type of traffic),text,"cpc, paid_social, event, referral, display, email, organic",str:1131,0,0.00,7,0
7,utm_campaign,UTM campaign tag = slug of campaign_name,text,lower_snake_case,str:1131,0,0.00,8,0
8,country,Country of the lead company,text,"Argentina, Chile, Uruguay, Brazil (full English name)",str:1131,0,0.00,13,79
9,city,City of the lead company,text,free text; may be empty,str:1078,53,4.69,11,0


In [ ]:
# ---- 2.5 Second view of the dictionary: observed values, ranges and format problems ----------------
# (same `data_dictionary` table, the detail columns that did not fit in the first view)
data_dictionary[["column", "value_range", "allowed_values_found", "examples", "format_violations", "violation_rule"]]

## 3. Data lineage
Where the leads data is born, how it moves and changes, and where it ends up.

In [ ]:
# =====================================================================
# SECTION 3 - DATA LINEAGE  (where the data is born, how it moves, how it changes, where it ends up)
# The pipeline stages below are taken from the project code (data_generation/gen_leads.py, common.py,
# airflow2/dags/b2b_daily_pipeline.py, requirements.md). The "evidence" column is measured from the files.
# =====================================================================

# ---- 3.1 End-to-end lineage of the leads data ---------------------------------------------------
lineage_stages = pd.DataFrame([
    # stage, component, input, transformation, output, status
    (1, "common.build_leads(day)", "Reference data (seeded) + campaign list + random generator seeded per day",
        "Creates the clean leads of that day: 'Won' leads (companies that later become buyers) + prospects that never convert",
        "In-memory clean lead records", "implemented"),
    (2, "gen_leads.messy()", "Clean lead records",
        "Injects real-world dirt: casing changes, typos in company_name, text dates dd/mm/yyyy, '$' strings, "
        "N/A or out-of-range lead_score, missing city/email/phone, future dates, duplicate rows",
        "Dirty lead rows", "implemented"),
    (3, "gen_leads.py (pandas + openpyxl)", "Dirty lead rows",
        "Writes one Excel workbook per day with leads; ~3% of files get a junk title row so the header moves to row 3",
        f"{len(lead_files)} files  data/output/leads/leads_YYYY-MM-DD.xlsx", "implemented"),
    (4, "Airflow DAG b2b_daily_pipeline  (task upload_to_s3)", "data/output/leads/*.xlsx",
        "Copies the files unchanged to S3 (whole folder on backfill, only the run day's file on daily runs)",
        "S3 bucket, same key  leads/leads_YYYY-MM-DD.xlsx", "implemented"),
    (5, "Databricks Auto Loader -> staging Delta tables (Unity Catalog)", "S3 / Unity Catalog Volume files",
        "Planned: ingest raw, keep every value as text, detect the header row, add load metadata", "Raw (staging) lead table", "planned (requirements.md)"),
    (6, "Warehouse transforms (MERGE, star schema)", "Raw lead table",
        "Planned: dedupe lead_id, parse dates, initcap source, country -> ISO-2, strip '$', lead_score 1-100 else NULL, "
        "lower/trim e-mail", "Lead dimension / fact in the warehouse", "planned (requirements.md)"),
    (7, "KPI views K11 / K12", "Clean lead table + customers + orders",
        "Lead conversion funnel (K11) and lead quality vs order value (K12)", "Business reports", "planned (requirements.md)"),
    (8, "THIS NOTEBOOK (scripts/notebook.ipynb)", "data/output/leads/*.xlsx",
        "Read-only exploration: merge all files into one DataFrame for EDA", "leads_raw (and df) in memory", "implemented"),
], columns=["stage", "component", "input", "transformation", "output", "status"]).set_index("stage")

# Evidence measured from the files for the implemented stages
evidence = {
    3: f"{len(lead_files)} files, {len(leads_raw):,} rows, {int((file_info['header_on_excel_row'] > 1).sum())} files with a title row",
    4: "no way to check S3 from here - the files on disk are what gets uploaded",
    8: f"leads_raw = {leads_raw.shape[0]:,} rows x {len(LEAD_COLUMNS)} business columns + 2 lineage columns",
}
lineage_stages["evidence_in_data"] = [evidence.get(i, "") for i in lineage_stages.index]

print("3.1 END-TO-END LINEAGE")
print("gen_leads.py -> Excel files -> Airflow upload_to_s3 -> S3 -> (planned) Auto Loader staging -> warehouse -> KPI views K11/K12\n")
pd.set_option("display.max_colwidth", 110)
pd.set_option("display.width", 250)
lineage_stages

In [ ]:
# ---- 3.2 Column-level lineage: how each column is produced, corrupted, and what we actually see ----
# origin = how common.build_leads() creates the value; corruption = what gen_leads.messy() may do to it;
# observed_violations = measured in the real files (from the data dictionary of section 2)
COLUMN_LINEAGE = {
    "lead_id":                ("LD-<creation date>-<running number of the day>", "duplicated rows (~2%)"),
    "created_date":           ("the day the file is built for", "5% text dd/mm/yyyy; 1% shifted 60-400 days into the future"),
    "source":                 ("2nd element of the campaign tuple", "8% UPPER / lower / padded with spaces"),
    "campaign_name":          ("1st element of the campaign tuple", "never corrupted"),
    "utm_source":             ("slug(source)", "none"),
    "utm_medium":             ("3rd element of the campaign tuple", "none"),
    "utm_campaign":           ("slug(campaign_name)", "none"),
    "country":                ("country of the buyer company (Won) or random city list (prospects)", "8% ISO code / UPPER / lower"),
    "city":                   ("city of the company", "4% set to empty"),
    "company_name":           ("buyer company name (Won) or random name + legal suffix", "8% letter dropped (typo); 5% UPPER / dots removed / trailing spaces"),
    "company_size":           ("size factor binned: 1-10 / 11-50 / 51-200 / 201-500 / 500+", "none"),
    "industry":               ("industry of the company", "none"),
    "lead_score":             ("Won: 55 + 20*ln(size) +/- noise; others ~ N(40,18); clipped to 1-100", "5% None / 'N/A' / empty; 2% out of range (150, -5, 999)"),
    "estimated_order_volume": ("5000 * size factor * random(0.6-1.4)", "5% empty; 5% text like '$4,321'"),
    "status":                 ("Won (converted company) or New/Contacted/Qualified/Proposal/Lost by weights 30/25/15/10/20", "none"),
    "utm_content":            ("random banner_a / banner_b / text_ad / empty", "none (about 25% empty by design)"),
    "contact_name":           ("random first + last name (Won: the real customer's name)", "none"),
    "contact_email":          ("first.last@company.com (Won: the customer's login e-mail)", "6% UPPER/empty; 3% '@' removed (non-Won only)"),
    "phone":                  ("country dial code + random digits (Won: the customer's phone)", "6% spaces/hyphens removed or empty"),
    "job_title":              ("random of 6 titles (Won: the customer's title)", "none"),
    "sales_rep":              ("random of 4 reps", "none"),
    "last_contacted_date":    ("creation day + 0-10 days (capped at today); empty for status New", "none"),
}

column_lineage = pd.DataFrame(
    [(c, *COLUMN_LINEAGE.get(c, ("?", "?"))) for c in LEAD_COLUMNS],
    columns=["column", "origin (build_leads)", "injected corruption (messy)"],
).merge(
    data_dictionary[["column", "format_violations", "null"]].rename(
        columns={"format_violations": "observed_format_violations", "null": "observed_nulls"}),
    on="column", how="left",
).set_index("column")

print("3.2 COLUMN-LEVEL LINEAGE")
column_lineage

In [ ]:
# ---- 3.3 Row-level lineage: every row can be traced back to its file and Excel row ---------------
# leads_raw carries _source_file and _source_row, so any record can be traced to its origin cell.
# Below: the files that deviate from the normal layout, and how a sample row traces back.
print("3.3 FILES THAT DEVIATE FROM THE NORMAL LAYOUT (header not on Excel row 1)")
display_cols = ["file", "file_date", "header_on_excel_row", "data_rows", "size_kb", "last_modified"]
print(file_info.loc[file_info["header_on_excel_row"] != 1, display_cols].to_string(index=False))

print("\n3.3b FILES PER YEAR-MONTH (when each batch was created)")
print(file_info.groupby(file_info["file_date"].dt.to_period("M")).agg(
    files=("file", "count"), rows=("data_rows", "sum")).T.to_string())

print("\n3.3c EXAMPLE OF ROW-LEVEL TRACEABILITY (3 random rows)")
leads_raw.sample(3, random_state=1)[["lead_id", "_source_file", "_source_row"]]

## 4. Data mapping
How leads fields correspond to fields in other datasets, tested wherever the leads data alone allows it.

In [ ]:
# =====================================================================
# SECTION 4 - DATA MAPPING  (how leads fields correspond to fields in other datasets / standard forms)
# 4.1 is the documented map (project docs + generator). 4.2 - 4.4 TEST the mappings that can be tested
# using the leads data alone. The Postgres tables (customers, companies) are not on disk here, so
# the cross-dataset match rates quoted in the docs cannot be re-measured in this notebook.
# =====================================================================

# ---- 4.1 Field mapping table ----------------------------------------------------------------------
field_mapping = pd.DataFrame([
    # leads field, target dataset.field, rule, strength
    ("contact_email", "Postgres customers.email", "lower(trim(contact_email)) = customers.email  (only status = 'Won')", "PRIMARY key - docs: 100% of Won leads match"),
    ("company_name", "Postgres companies.name", "cleaned name (trim, upper/lower, dots removed) = companies.name", "FALLBACK - docs: 93% match (typos break it)"),
    ("country", "Postgres companies.country", "map ISO code / lower / UPPER to the full English name (AR -> Argentina ...)", "conform before joining"),
    ("utm_campaign", "Web logs: utm_campaign in landing URL", "same slug of the campaign name", "traffic attribution only (never revenue)"),
    ("utm_source / utm_medium", "Web logs: utm_source / utm_medium in landing URL", "same slugs", "traffic attribution only"),
    ("campaign_name", "utm_campaign (same file)", "utm_campaign = slug(campaign_name)", "internal consistency"),
    ("source", "utm_source (same file)", "utm_source = slug(source)", "internal consistency"),
    ("lead_id", "(none outside leads)", "primary key; dedupe on it", "no cross-dataset use"),
    ("created_date", "Postgres companies.created_at", "Won lead is created 5-25 days BEFORE the company's created_at", "lead-to-first-order lag (K11)"),
    ("lead_score / estimated_order_volume", "Postgres orders / order_items (via Won chain)", "lead -> email -> customer -> company -> orders", "K12: score vs actual order value"),
], columns=["leads_field", "target_dataset.field", "mapping_rule", "strength / documented match"])
print("4.1 FIELD MAPPING (leads -> other datasets)")
pd.set_option("display.max_colwidth", 95)
pd.set_option("display.width", 250)
display(field_mapping)

# ---- 4.2 Internal mapping check: utm tags vs source / campaign_name -------------------------------
def slug(s):                                           # same rule as data_generation/common.slug
    return re.sub(r"\W+", "_", str(s).lower())

clean_source = leads_raw["source"].dropna().astype(str).str.strip()
chk = pd.DataFrame({
    "utm_campaign == slug(campaign_name)": (leads_raw["utm_campaign"].astype(str) == leads_raw["campaign_name"].map(slug)),
    "utm_source == slug(source)  (raw source)": (leads_raw["utm_source"].astype(str) == leads_raw["source"].map(slug)),
    "utm_source == slug(source)  (source trimmed)": (leads_raw["utm_source"].astype(str) == leads_raw["source"].astype(str).str.strip().map(slug)),
})
print("4.2 DO THE UTM TAGS FOLLOW THE SOURCE / CAMPAIGN?  (share of rows that agree, %)")
print((chk.mean() * 100).round(2).to_string(), "\n")

# ---- 4.3 campaign_name -> source -> utm_medium: is the mapping one-to-one? ----------------------
campaign_map = (leads_raw.assign(source=leads_raw["source"].astype(str).str.strip().str.title())
                .groupby("campaign_name")
                .agg(rows=("lead_id", "size"),
                     sources=("source", lambda x: sorted(x.unique())),
                     utm_mediums=("utm_medium", lambda x: sorted(x.dropna().unique())),
                     utm_campaign=("utm_campaign", lambda x: sorted(x.dropna().unique()))))
print("4.3 CAMPAIGN -> SOURCE -> MEDIUM MAPPING (as observed)")
display(campaign_map)

# ---- 4.4 country mapping: every raw spelling -> the standard full name and ISO-2 -----------------
COUNTRY_TO_ISO = {"argentina": "AR", "chile": "CL", "uruguay": "UY", "brazil": "BR"}   # from common.COUNTRIES
ISO_TO_COUNTRY = {v.lower(): k.title() for k, v in COUNTRY_TO_ISO.items()}
def standard_country(v):
    key = str(v).strip().lower()
    return ISO_TO_COUNTRY.get(key) or (key.title() if key in COUNTRY_TO_ISO else None)

country_map = (leads_raw["country"].dropna().astype(str).value_counts().rename("rows").reset_index()
               .rename(columns={"country": "raw_value"}))
country_map["maps_to_country"] = country_map["raw_value"].map(standard_country)
country_map["iso2"] = country_map["maps_to_country"].str.lower().map(COUNTRY_TO_ISO)
print("4.4 COUNTRY: RAW SPELLINGS -> STANDARD VALUE")
print(f"Raw values that cannot be mapped: {int(country_map['maps_to_country'].isna().sum())}")
display(country_map)

# ---- 4.5 How many leads can actually be joined downstream? ------------------------------------
email = leads_raw["contact_email"].dropna().astype(str).str.strip().str.lower()
valid_email = email.str.match(EMAIL_RE)
won = leads_raw["status"].astype(str).str.strip().str.lower() == "won"
join_ready = pd.Series({
    "Rows": len(leads_raw),
    "Won rows (only these can join to customers)": int(won.sum()),
    "Won rows with a usable e-mail (valid after lower/trim)": int((won & leads_raw["contact_email"].notna()
                                                                 & leads_raw["contact_email"].astype(str).str.strip().str.lower().str.match(EMAIL_RE)).sum()),
    "ALL rows with a valid e-mail after lower/trim": int(valid_email.sum()),
    "Rows with empty or invalid e-mail": int(len(leads_raw) - valid_email.sum()),
})
print("4.5 JOIN READINESS")
print(join_ready.to_string())

## 5. Business glossary
Plain-language definitions of every business term, with the live number from the data.

In [ ]:
# =====================================================================
# SECTION 5 - BUSINESS GLOSSARY  (plain-language meaning of every term, with the live number from the data)
# Definitions come from the project docs / generator. The "in this data" column is calculated from leads_raw
# after a light, in-memory clean-up (leads_raw itself is never changed): duplicates on lead_id are counted once.
# =====================================================================

# ---- 5.1 Temporary clean view used ONLY to compute the numbers shown in the glossary -----------
g = leads_raw.drop_duplicates(subset="lead_id").copy()                 # one row per lead
g["status_c"] = g["status"].astype(str).str.strip().str.title()
g["source_c"] = g["source"].astype(str).str.strip().str.title()
g["score_n"] = g["lead_score"].map(to_number)
g.loc[(g["score_n"] < 1) | (g["score_n"] > 100), "score_n"] = float("nan")   # out of range = unusable
g["volume_n"] = g["estimated_order_volume"].map(to_number)
g["created_c"] = g["created_date"].map(to_date)

n_leads = len(g)
n_won = int((g["status_c"] == "Won").sum())
by_status = g["status_c"].value_counts()


def pct(a, b):
    return f"{a / b * 100:.1f}%" if b else "n/a"


# ---- 5.2 Glossary -----------------------------------------------------------------------------
glossary = pd.DataFrame([
    ("Lead", "A company/person that showed interest and was captured by a marketing activity; the unit of this dataset",
     "One row per lead_id", f"{n_leads:,} distinct leads ({len(leads_raw):,} rows incl. duplicates)"),
    ("Won lead (converted lead)", "A lead that became a buying customer. Only Won leads can be linked to real orders",
     "status = 'Won'", f"{n_won:,} Won leads = {pct(n_won, n_leads)} of leads"),
    ("Lead status / funnel stage", "Where the lead was in the sales funnel. It is a snapshot taken at creation and is never updated, so there is no stage history",
     "status: New, Contacted, Qualified, Proposal, Lost, Won", ", ".join(f"{k} {v}" for k, v in by_status.items())),
    ("Lead conversion rate", "Share of leads that became Won", "Won leads / distinct leads",
     pct(n_won, n_leads)),
    ("Source (channel)", "The marketing channel the lead came from (Google Ads, LinkedIn, Trade Show, Referral, Facebook, Email, Organic)",
     "source (cleaned with trim + title case)", f"{g['source_c'].nunique()} channels, largest: {g['source_c'].value_counts().idxmax()}"),
    ("Campaign", "A specific marketing initiative inside a channel; the most reliable key for campaign analysis because it is never corrupted",
     "campaign_name", f"{g['campaign_name'].nunique()} campaigns, largest: {g['campaign_name'].value_counts().idxmax()}"),
    ("UTM parameters", "Tags added to campaign links so web traffic can be traced to its source (utm_source), type (utm_medium), campaign (utm_campaign) and ad variant (utm_content)",
     "utm_source, utm_medium, utm_campaign, utm_content", f"{g['utm_medium'].nunique()} mediums: {', '.join(sorted(g['utm_medium'].dropna().unique()))}"),
    ("Lead score", "A 1-100 rating of lead quality; higher means a better, more promising lead",
     "lead_score (valid only 1-100)", f"usable for {g['score_n'].notna().sum():,} leads; mean {g['score_n'].mean():.1f}, median {g['score_n'].median():.0f}"),
    ("Estimated order volume", "The order value the lead is expected to bring (currency units); an estimate, not a real order",
     "estimated_order_volume", f"median {g['volume_n'].median():,.0f}, mean {g['volume_n'].mean():,.0f} (from {g['volume_n'].notna().sum():,} usable values)"),
    ("Company size", "Employee-count band of the prospect: 1-10, 11-50, 51-200, 201-500, 500+",
     "company_size", ", ".join(f"{k} {v}" for k, v in g["company_size"].value_counts().items())),
    ("Industry", "Business sector of the prospect company", "industry",
     f"{g['industry'].nunique()} industries, largest: {g['industry'].value_counts().idxmax()}"),
    ("Sales rep", "The salesperson who owns and follows up the lead", "sales_rep",
     f"{g['sales_rep'].nunique()} reps: {', '.join(sorted(g['sales_rep'].dropna().unique()))}"),
    ("Contact", "The person at the prospect company we talk to (name, e-mail, phone, job title). For Won leads this is the real customer, so the e-mail is the join key",
     "contact_name, contact_email, phone, job_title", f"{g['contact_email'].notna().sum():,} leads have an e-mail"),
    ("Last contacted date", "Last day the sales rep got in touch; empty for leads still in status New",
     "last_contacted_date", f"empty for {g['last_contacted_date'].isna().sum():,} leads (New leads: {int(by_status.get('New', 0)):,})"),
    ("Lead creation date", "The day the lead was captured; it is also the date in the file name and in lead_id",
     "created_date", f"{g['created_c'].min().date()} to {g['created_c'].max().date()} (before date repair)"),
    ("Lead-sourced revenue (K11 / K12)", "Revenue from customers that first came in as a Won lead; reached via lead -> e-mail -> customer -> company -> orders",
     "Needs customers / orders tables (not in this dataset)", "cannot be computed from leads alone"),
    ("Prospect (open lead)", "A lead that has not been won or lost yet", "status in New, Contacted, Qualified, Proposal",
     f"{int(g['status_c'].isin(['New', 'Contacted', 'Qualified', 'Proposal']).sum()):,} leads"),
    ("Lost lead", "A lead that was closed without becoming a customer", "status = 'Lost'", f"{int(by_status.get('Lost', 0)):,} leads"),
], columns=["term", "business_definition", "how_it_is_stored", "in_this_data"]).set_index("term")

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 250)
glossary

## 6. Data catalog
Searchable inventory of the datasets, the leads columns and their quality, tied to the glossary.

In [ ]:
# =====================================================================
# SECTION 6 - DATA CATALOG  (searchable inventory: what datasets exist, where, what they hold, how they relate)
# It combines the metadata, lineage and glossary built above. The related datasets are the other
# files that sit in data/output next to the leads (counted from disk, read-only).
# =====================================================================
OUT_DIR = LEADS_DIR.parent                                             # data/output


def folder_stats(pattern_dir, pattern):
    """(number of files, size in MB) of a folder; Excel lock files (~$...) are ignored."""
    files = [f for f in pattern_dir.glob(pattern) if not f.name.startswith("~$")] if pattern_dir.exists() else []
    return len(files), round(sum(f.stat().st_size for f in files) / 1024**2, 2)


n_logs, mb_logs = folder_stats(OUT_DIR / "web_logs", "*.log")
n_leads_files, mb_leads = folder_stats(LEADS_DIR, "*.xlsx")

# ---- 6.1 Catalog of datasets --------------------------------------------------------------------
catalog = pd.DataFrame([
    ("leads", "Marketing leads exported from the marketing team, one workbook per day",
     "Excel (.xlsx)", str(LEADS_DIR), n_leads_files, mb_leads, f"{len(leads_raw):,} rows x {len(LEAD_COLUMNS)} cols",
     "1 row = 1 lead", "lead_id", "Daily (Airflow b2b_daily_pipeline)",
     "marketing, leads, campaign, utm, funnel, conversion, lead score, sales rep", "THIS DATASET - profiled in sections 1-5"),
    ("web_logs", "Web server access logs (Apache combined format) of the B2B shop",
     "Text logs (.log)", str(OUT_DIR / "web_logs"), n_logs, mb_logs, "(not loaded here)",
     "1 row = 1 HTTP request", "none", "Daily",
     "web, traffic, http, ip, user agent, utm, checkout", "Related by utm_campaign / utm_source and auth_user = customer e-mail"),
    ("ip_geo_lookup", "Lookup from the first octet of an IP address to a country code",
     "CSV", str(OUT_DIR / "ip_geo_lookup.csv"), int((OUT_DIR / "ip_geo_lookup.csv").exists()), 0, "(not loaded here)",
     "1 row = 1 IP first octet", "first_octet", "Static reference", "geo, ip, country", "Used with web_logs, not with leads"),
    ("bad_rows_manifest", "Ground truth list of deliberately corrupted rows in the PostgreSQL data",
     "CSV", str(OUT_DIR / "bad_rows_manifest.csv"), int((OUT_DIR / "bad_rows_manifest.csv").exists()), 0, "(not loaded here)",
     "1 row = 1 corrupted value", "table + primary_key + column", "Per backfill", "data quality, manifest",
     "Covers PostgreSQL only - nothing for leads"),
    ("postgres (companies, customers, orders, order_items, products, suppliers, ...)", "Operational B2B database",
     "PostgreSQL (copied to Neon)", "Docker postgres -> Neon (Airflow task sync_to_neon)", None, None, "(not on disk)",
     "1 row = company / customer / order line / ...", "table primary keys", "Daily (CDC by updated_at)",
     "orders, revenue, customers, companies, products", "Won leads join to customers.email -> company -> orders"),
], columns=["dataset", "description", "format", "location", "files", "size_mb", "volume", "grain", "key",
            "refresh", "tags", "relationship_to_leads"]).set_index("dataset")

catalog["files"] = catalog["files"].astype("Int64")                    # whole numbers, <NA> where not on disk
print("6.1 DATA CATALOG")
pd.set_option("display.max_colwidth", 70)
pd.set_option("display.width", 250)
display(catalog)

# ---- 6.2 Catalog entry for the leads dataset: column inventory with quality flags --------------------
column_catalog = data_dictionary[["column", "meaning", "expected_type", "null_%", "distinct", "format_violations"]].copy()
column_catalog["quality"] = column_catalog.apply(
    lambda r: "OK" if r["format_violations"] == 0 and r["null_%"] == 0 else
              ("NEEDS CLEANING" if r["format_violations"] > 0 else "HAS EMPTIES"), axis=1)
column_catalog["glossary_terms"] = column_catalog["column"].map(
    lambda c: ", ".join(t for t, row in glossary.iterrows() if re.search(rf"\b{re.escape(c)}\b", row["how_it_is_stored"])))
print("6.2 COLUMN CATALOG FOR LEADS")
display(column_catalog)


# ---- 6.3 Search the catalog -----------------------------------------------------------------------
def search_catalog(keyword):
    """Case-insensitive search over datasets, columns and glossary terms. Example: search_catalog('email')."""
    k = keyword.lower()
    hits = {
        "datasets": catalog[catalog.apply(lambda r: k in (r.name + " " + " ".join(map(str, r.values))).lower(), axis=1)].index.tolist(),
        "columns": column_catalog[column_catalog.apply(lambda r: k in " ".join(map(str, r.values)).lower(), axis=1)]["column"].tolist(),
        "glossary terms": glossary[glossary.apply(lambda r: k in (r.name + " " + " ".join(map(str, r.values))).lower(), axis=1)].index.tolist(),
    }
    return hits


# Example: change the keyword to search for anything
search_catalog("email")

,lead_id,created_date,source,campaign_name,utm_source,utm_medium,utm_campaign,country,city,company_name,...,lead_score,estimated_order_volume,status,utm_content,contact_name,contact_email,phone,job_title,sales_rep,last_contacted_date
0,LD-20241105-001,05/11/2024,Google Ads,Retargeting Display,google_ads,display,retargeting_display,Argentina,Buenos Aires,Mercurio Textil S.R.L.,...,51,10859,Contacted,banner_b,Lucia Romero,lucia.romero@mercuriotextil.com,+54 31 2237-5211,Operations Director,Luis Benitez,2024-11-14
1,LD-20241105-001,05/11/2024,Google Ads,Retargeting Display,google_ads,display,retargeting_display,Argentina,Buenos Aires,Mercurio Textil S.R.L.,...,51,10859,Contacted,banner_b,Lucia Romero,lucia.romero@mercuriotextil.com,+54 31 2237-5211,Operations Director,Luis Benitez,2024-11-14
2,LD-20241105-002,2024-11-05 00:00:00,Email,Newsletter Promo,email,email,newsletter_promo,Argentina,La Plata,Rio Transportes S.R.L.,...,43,13915,Contacted,text_ad,Diego Acosta,diego.acosta@riotransportes.com,+54 72 6416-1666,Office Manager,Ana Torres,2024-11-15


In [25]:
df.isna().sum()

lead_id                     63
created_date                63
source                      63
campaign_name               63
utm_source                  63
utm_medium                  63
utm_campaign                63
country                     63
city                       115
company_name                63
company_size                63
industry                    63
lead_score                 118
estimated_order_volume      88
status                      63
utm_content                341
contact_name                63
contact_email               92
phone                      100
job_title                   63
sales_rep                   63
last_contacted_date        346
Marketing leads export    1113
Unnamed: 1                1113
Unnamed: 2                1113
Unnamed: 3                1113
Unnamed: 4                1113
Unnamed: 5                1113
Unnamed: 6                1113
Unnamed: 7                1113
Unnamed: 8                1114
Unnamed: 9                1113
Unnamed:

In [ ]:
df.